# Notebook 1: Document Embeddings for Topic Modeling

This notebook generates document-level embeddings for ~6,000 Common Crawl web pages using transformer-based models. These embeddings are the **first step of the BERTopic pipeline** and will be used in Notebook 2 for topic discovery.

We work with three embedding models:

| Model | Dimensions | Source | Cost | Notes |
|-------|-----------|--------|------|-------|
| `all-MiniLM-L6-v2` | 384 | Sentence Transformers | Free (local) | **BERTopic's default** |
| OpenAI `text-embedding-3-small` | 1,536 | OpenAI API | ~$0.06 | Commercial API |
| Google `text-embedding-004` | 768 | Google AI Studio | Free (API) | Free tier with rate limits |

**For your assignment**, you will also choose a **fourth model of your own** from HuggingFace. Section 5 of this notebook guides you through that selection process.

**Outputs:** Saved embedding matrices (`.npy` files) ready for Notebook 2.

---

## 1. Setup & Installation

In [1]:
# Install required packages (uncomment as needed)
# !pip install sentence-transformers openai google-generativeai pandas numpy scikit-learn tqdm matplotlib seaborn tiktoken

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
import os

warnings.filterwarnings('ignore')
print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2. Load Data

In [3]:
# -------------------------------------------------------------------
# UPDATE THIS PATH to point to your full ~6000 page dataset
# -------------------------------------------------------------------
DATA_PATH = "../data/english_pages_metadata_clean_with_labels.csv"

df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
df['full_text'] = df['full_text'].fillna('').astype(str)

print(f"Loaded {len(df)} pages")
print(f"Columns: {list(df.columns)}")

# Text length statistics
print(f"\nText length statistics (characters):")
print(df['full_text'].str.len().describe())

df.head(3)

Loaded 596 pages
Columns: ['page_id', 'assigned_to', 'manual_label', 'manual_label_clean', 'manual_label_final', 'full_text', '_merge']

Text length statistics (characters):
count      596.000000
mean      6535.971477
std       9110.613565
min        294.000000
25%       1748.500000
50%       3446.500000
75%       7771.000000
max      75582.000000
Name: full_text, dtype: float64


,page_id,assigned_to,manual_label,manual_label_clean,manual_label_final,full_text,_merge
0,655.0,Vinay Varshigan Sivakumar Jayalakshmi,ecommerce,ECOMMERCE,ECOMMERCE,Fall arrest and work positioning harness – All...,both
1,656.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Vermont Mountain Eats: Jay Peak - All Mountain...,both
2,657.0,Vinay Varshigan Sivakumar Jayalakshmi,news,NEWS,NEWS,05/18/2020 Booking Report for Bulloch County -...,both


In [5]:
# Load labels if available (for downstream evaluation in Notebook 2)
LABELS_PATH = "../data/english_pages_metadata_clean_with_labels.csv"

if os.path.exists(LABELS_PATH):
    labels_df = pd.read_csv(LABELS_PATH)
    labels_subset = labels_df[['page_id', 'manual_label_final']].copy()
    df = pd.merge(df, labels_subset, on='page_id', how='left')
    print(f"Labels merged. Distribution:")
    print(df['manual_label_final'].value_counts())
else:
    print(f"Labels file not found. Proceeding without labels.")

Labels merged. Distribution:
manual_label_final
OTHER               158
BLOG                123
ECOMMERCE           114
EDUCATION            61
NEWS                 47
FORUM/DISCUSSION     46
TECHNICAL            34
GOVERNMENT           13
Name: count, dtype: int64


In [6]:
# Prepare text list used by all models
texts = df['full_text'].tolist()
print(f"Prepared {len(texts)} documents for embedding.")

Prepared 596 documents for embedding.


---
# Part A: `all-MiniLM-L6-v2` (384-D, BERTopic's default)
---

This is the model BERTopic uses by default. It is fast, lightweight (~80MB), and produces 384-dimensional embeddings. A solid baseline for topic modeling.

**Max sequence length:** 256 word pieces. Longer documents are truncated.

In [7]:
from sentence_transformers import SentenceTransformer

mini_model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"Model loaded: all-MiniLM-L6-v2")
print(f"Max sequence length: {mini_model.max_seq_length}")
print(f"Embedding dimension: {mini_model.get_sentence_embedding_dimension()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: all-MiniLM-L6-v2
Max sequence length: 256
Embedding dimension: 384


In [8]:
print(f"Generating MiniLM embeddings for {len(texts)} documents...")
start_time = time.time()

mini_embeddings = mini_model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

elapsed = time.time() - start_time
print(f"\nDone in {elapsed:.1f} seconds ({len(texts)/elapsed:.1f} docs/sec)")
print(f"Embedding matrix shape: {mini_embeddings.shape}")

Generating MiniLM embeddings for 596 documents...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]


Done in 1.7 seconds (354.7 docs/sec)
Embedding matrix shape: (596, 384)


In [9]:
np.save('minilm_embeddings.npy', mini_embeddings)
print(f"Saved minilm_embeddings.npy, shape {mini_embeddings.shape}")

Saved minilm_embeddings.npy, shape (596, 384)


---
# Part B: OpenAI `text-embedding-3-small` (1,536-D)
---

Commercial embedding model from OpenAI. Higher-dimensional, longer context window (8,192 tokens), costs ~$0.06 for this corpus.

**Skip this section if you don't have an OpenAI API key.**

In [ ]:
from openai import OpenAI
import tiktoken

with open('openai_key.txt', 'r') as f:
    api_key = f.read().strip()

client = OpenAI(api_key=api_key)
OPENAI_MODEL = "text-embedding-3-small"

encoding = tiktoken.get_encoding("cl100k_base")
MAX_TOKENS = 8000

print(f"OpenAI client initialized. Model: {OPENAI_MODEL}")

OpenAI client initialized. Model: text-embedding-3-small


In [ ]:
def truncate_text(text, max_tokens=MAX_TOKENS):
    """Truncate text to fit within token limit."""
    tokens = encoding.encode(text)
    if len(tokens) > max_tokens:
        return encoding.decode(tokens[:max_tokens])
    return text

def get_openai_embeddings(texts, client, model=OPENAI_MODEL, batch_size=100):
    """Generate OpenAI embeddings in batches."""
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="OpenAI batches"):
        batch = texts[i : i + batch_size]
        batch = [t if t.strip() else " " for t in batch]
        response = client.embeddings.create(model=model, input=batch)
        all_embeddings.extend([item.embedding for item in response.data])
    return np.array(all_embeddings)

texts_strings = [" ".join(t) for t in texts]
oai_texts = [truncate_text(t) for t in texts_strings]

print(f"Generating OpenAI embeddings for {len(oai_texts)} documents...")
print(f"Estimated cost: ~${len(oai_texts) * 500 * 0.02 / 1_000_000:.2f}")
start_time = time.time()

oai_embeddings = get_openai_embeddings(oai_texts, client)

elapsed = time.time() - start_time
print(f"\nDone in {elapsed:.1f} seconds")
print(f"Embedding matrix shape: {oai_embeddings.shape}")

Generating OpenAI embeddings for 6314 documents...
Estimated cost: ~$0.06


OpenAI batches: 100%|██████████| 64/64 [08:37<00:00,  8.08s/it]



Done in 517.4 seconds
Embedding matrix shape: (6314, 1536)


In [ ]:
np.save('openai_embeddings.npy', oai_embeddings)
print(f"Saved openai_embeddings.npy, shape {oai_embeddings.shape}")

Saved openai_embeddings.npy, shape (6314, 1536)


---
# Part C: Google Gemini `text-embedding-004` (768-D)
---

Free embedding model from Google via AI Studio. 768 dimensions, 2,048 token limit.

Get your API key at [aistudio.google.com](https://aistudio.google.com)

**Skip this section if you don't have a Google AI Studio API key.**

In [ ]:
import google.generativeai as genai

with open('google_key.txt', 'r') as f:
    google_api_key = f.read().strip()

genai.configure(api_key=google_api_key)

GOOGLE_MODEL = "models/text-embedding-004"
GOOGLE_DIMS = 768

print(f"Google client initialized. Model: {GOOGLE_MODEL}, Dimensions: {GOOGLE_DIMS}")

ModuleNotFoundError: No module named 'google.generativeai'

In [ ]:
def get_google_embeddings(texts, model=GOOGLE_MODEL, batch_size=20, delay=2):
    """Generate Google Gemini embeddings in batches with rate limiting."""
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Google batches"):
        batch = texts[i : i + batch_size]
        batch = [t if t.strip() else " " for t in batch]
        try:
            result = genai.embed_content(
                model=model, content=batch, task_type="classification"
            )
            all_embeddings.extend(result['embedding'])
        except Exception as e:
            print(f"Error at batch {i}: {e}")
            all_embeddings.extend([[0.0] * GOOGLE_DIMS] * len(batch))
        if delay > 0:
            time.sleep(delay)
    return np.array(all_embeddings)

google_texts = [' '.join(t.split()[:1500]) for t in texts]

print(f"Generating Google embeddings for {len(google_texts)} documents...")
start_time = time.time()

google_embeddings = get_google_embeddings(google_texts)

elapsed = time.time() - start_time
print(f"\nDone in {elapsed:.1f} seconds")
print(f"Embedding matrix shape: {google_embeddings.shape}")

In [ ]:
np.save('google_embeddings.npy', google_embeddings)
print(f"Saved google_embeddings.npy, shape {google_embeddings.shape}")

---
# Part D: Token-Level Sanity Checks
---

Before trusting document embeddings, verify the models capture meaningful semantic relationships at the word level.

In [ ]:
# Test words organized by expected category
test_words = {
    'ECOMMERCE':  ['price', 'shipping', 'cart', 'buy', 'product', 'discount'],
    'NEWS':       ['reporter', 'headline', 'breaking', 'journalist', 'politics'],
    'EDUCATION':  ['university', 'student', 'course', 'professor', 'curriculum'],
    'GOVERNMENT': ['government', 'legislation', 'federal', 'policy', 'regulation'],
    'TECHNICAL':  ['algorithm', 'software', 'database', 'API', 'debugging'],
    'BLOG':       ['opinion', 'personal', 'lifestyle', 'travel', 'diary'],
    'FORUM':      ['thread', 'reply', 'moderator', 'upvote', 'discussion'],
}

all_test_words = []
word_categories = []
for cat, words in test_words.items():
    for w in words:
        all_test_words.append(w)
        word_categories.append(cat)

# Encode with MiniLM
mini_word_embs = mini_model.encode(all_test_words, normalize_embeddings=True)

# Cosine similarity heatmap
sim_matrix = cosine_similarity(mini_word_embs)

plt.figure(figsize=(14, 12))
sns.heatmap(sim_matrix, xticklabels=all_test_words, yticklabels=all_test_words,
            cmap='RdYlBu_r', vmin=-0.1, vmax=1.0, annot=False)
plt.title('MiniLM (384-D): Token Cosine Similarities', fontsize=14)
plt.xticks(rotation=90, fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

print("Look for block-diagonal structure: words within the same category")
print("should be more similar (warmer colors) than words across categories.")

## Document-Level Comparison

In [ ]:
# Compare embedding statistics across available models
models = {'MiniLM (384-D)': mini_embeddings}
if 'oai_embeddings' in dir(): models['OpenAI (1536-D)'] = oai_embeddings
if 'google_embeddings' in dir(): models['Google (768-D)'] = google_embeddings

print(f"{'Model':<22} {'Shape':>15} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 80)
for name, embs in models.items():
    print(f"{name:<22} {str(embs.shape):>15} {embs.mean():>10.6f} {embs.std():>10.6f} {embs.min():>10.4f} {embs.max():>10.4f}")

In [ ]:
# Inter-document cosine similarity distributions
np.random.seed(42)
sample_idx = np.random.choice(len(df), size=min(500, len(df)), replace=False)

fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 5))
if len(models) == 1: axes = [axes]

colors = ['steelblue', 'darkorange', 'mediumpurple']
for idx, (name, embs) in enumerate(models.items()):
    sample_sims = cosine_similarity(embs[sample_idx])
    triu = np.triu_indices_from(sample_sims, k=1)
    pairwise = sample_sims[triu]
    
    axes[idx].hist(pairwise, bins=50, alpha=0.7, color=colors[idx % len(colors)], edgecolor='black')
    axes[idx].set_title(f'{name}')
    axes[idx].set_xlabel('Cosine Similarity')
    axes[idx].set_ylabel('Frequency')
    axes[idx].axvline(pairwise.mean(), color='red', linestyle='--', label=f'Mean: {pairwise.mean():.3f}')
    axes[idx].legend()

plt.suptitle('Inter-Document Cosine Similarity Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Wider spread = more discriminative embeddings (better for clustering).")

In [ ]:
# Nearest neighbor sanity check
def find_nearest(query_idx, embeddings, df, top_n=5):
    query = embeddings[query_idx].reshape(1, -1)
    sims = cosine_similarity(query, embeddings)[0]
    sims[query_idx] = -1
    top = np.argsort(sims)[-top_n:][::-1]
    return [{'page_id': df.iloc[i]['page_id'],
             'label': df.iloc[i].get('manual_label_final', 'N/A'),
             'sim': sims[i],
             'text': str(df.iloc[i]['full_text'])[:100]} for i in top]

label_col = 'manual_label_final'
if label_col in df.columns:
    sample_cats = list(df[df[label_col].notna()][label_col].unique())[:4]
    for cat in sample_cats:
        idx = df[df[label_col] == cat].index[0]
        print(f"\n{'='*70}")
        print(f"Query: page_id={df.iloc[idx]['page_id']}, label={cat}")
        for name, embs in models.items():
            print(f"  {name}:")
            for r in find_nearest(idx, embs, df, top_n=3):
                match = 'Y' if r['label'] == cat else 'N'
                print(f"    [{match}] page={r['page_id']}, label={r['label']}, sim={r['sim']:.4f}")

---
# Part E: Choose Your Own Model (Assignment Requirement)
---

For your assignment, you must select **one additional embedding model** from HuggingFace and run it through the same pipeline. This section guides you through the selection and implementation.

### How to choose a model

Use what you learned in Notebook 0 (HuggingFace Exploration) to make an informed choice. Consider:

1. **MTEB clustering score**: higher score = likely better topic modeling results
2. **Dimensions**: 384 (fast) vs 768 (balanced) vs 1024+ (highest quality but slower)
3. **Max sequence length**: 256 tokens (MiniLM) vs 512 vs 8192 (for long web pages)
4. **Model size**: affects download time and memory usage
5. **Specific reasoning**: maybe you want a multilingual model, a domain-specific model, or one with a very long context window

### Some strong candidates to consider

| Model | Dims | Max Tokens | Why consider it? |
|-------|------|-----------|------------------|
| `BAAI/bge-base-en-v1.5` | 768 | 512 | Top MTEB clustering scores |
| `BAAI/bge-small-en-v1.5` | 384 | 512 | High quality at MiniLM's size |
| `intfloat/e5-base-v2` | 768 | 512 | Strong all-around performer |
| `thenlper/gte-base` | 768 | 512 | Competitive alternative |
| `jinaai/jina-embeddings-v2-base-en` | 768 | 8192 | Very long context window |
| `nomic-ai/nomic-embed-text-v1.5` | 768 | 8192 | Long context + strong quality |
| `sentence-transformers/all-mpnet-base-v2` | 768 | 384 | Best open-source sentence transformer |
| `mixedbread-ai/mxbai-embed-large-v1` | 1024 | 512 | Among the highest MTEB scores |

You are **not limited to this list**. You may choose any model from HuggingFace that is compatible with the `sentence-transformers` library.

### Your justification (required in your assignment submission)

In a markdown cell, write 3-5 sentences explaining:
- Which model you chose and why
- What you expect compared to MiniLM (better quality? longer context? different trade-off?)
- What MTEB scores or other data informed your decision

In [ ]:
# ============================================================
# YOUR MODEL CHOICE
# Replace the model name below with your chosen model
# ============================================================

YOUR_MODEL_NAME = 'BAAI/bge-base-en-v1.5'  # <-- CHANGE THIS

print(f"Loading your chosen model: {YOUR_MODEL_NAME}")
print("(This may take a minute to download on first run)\n")

your_model = SentenceTransformer(YOUR_MODEL_NAME)

print(f"Model loaded: {YOUR_MODEL_NAME}")
print(f"Max sequence length: {your_model.max_seq_length}")
print(f"Embedding dimension: {your_model.get_sentence_embedding_dimension()}")

In [ ]:
# Generate embeddings with your chosen model
print(f"Generating embeddings with {YOUR_MODEL_NAME}...")
start_time = time.time()

your_embeddings = your_model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

elapsed = time.time() - start_time
print(f"\nDone in {elapsed:.1f} seconds ({len(texts)/elapsed:.1f} docs/sec)")
print(f"Embedding matrix shape: {your_embeddings.shape}")

In [ ]:
# Save your model's embeddings
# Create a clean filename from the model name
safe_name = YOUR_MODEL_NAME.replace('/', '_').replace('-', '_')
your_save_path = f'{safe_name}_embeddings.npy'

np.save(your_save_path, your_embeddings)
print(f"Saved {your_save_path}, shape {your_embeddings.shape}")

In [ ]:
# Token-level sanity check for your model
your_word_embs = your_model.encode(all_test_words, normalize_embeddings=True)
your_sim = cosine_similarity(your_word_embs)

# Side-by-side comparison: MiniLM vs your model
fig, axes = plt.subplots(1, 2, figsize=(20, 9))

sns.heatmap(sim_matrix, xticklabels=all_test_words, yticklabels=all_test_words,
            cmap='RdYlBu_r', vmin=-0.1, vmax=1.0, annot=False, ax=axes[0])
axes[0].set_title('MiniLM (384-D)', fontsize=13)
axes[0].tick_params(axis='both', labelsize=7)

sns.heatmap(your_sim, xticklabels=all_test_words, yticklabels=all_test_words,
            cmap='RdYlBu_r', vmin=-0.1, vmax=1.0, annot=False, ax=axes[1])
axes[1].set_title(f'{YOUR_MODEL_NAME}', fontsize=13)
axes[1].tick_params(axis='both', labelsize=7)

plt.suptitle('Token Similarity Comparison: MiniLM vs. Your Model', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("Compare the block-diagonal structure between the two models.")
print("Does your model produce tighter within-category clusters?")

In [ ]:
# Document-level comparison: add your model to the comparison
models[f'Your choice ({your_embeddings.shape[1]}-D)'] = your_embeddings

print(f"\n{'Model':<30} {'Shape':>15} {'Mean':>10} {'Std':>10}")
print("-" * 70)
for name, embs in models.items():
    print(f"{name:<30} {str(embs.shape):>15} {embs.mean():>10.6f} {embs.std():>10.6f}")

In [ ]:
# Nearest neighbor comparison including your model
if label_col in df.columns:
    cat = list(df[df[label_col].notna()][label_col].unique())[0]
    idx = df[df[label_col] == cat].index[0]
    print(f"Query: page_id={df.iloc[idx]['page_id']}, label={cat}")
    print(f"Text: {str(df.iloc[idx]['full_text'])[:120]}...\n")
    
    for name, embs in models.items():
        print(f"  {name}:")
        for r in find_nearest(idx, embs, df, top_n=3):
            match = 'Y' if r['label'] == cat else 'N'
            print(f"    [{match}] page={r['page_id']}, label={r['label']}, sim={r['sim']:.4f}")
        print()

---
# Save Everything for Notebook 2
---

In [ ]:
# Save page IDs for alignment in Notebook 2
df[['page_id']].to_csv('page_ids.csv', index=False)

print("\n" + "=" * 60)
print("SAVED FILES")
print("=" * 60)
print(f"  page_ids.csv              : {len(df)} page IDs")
print(f"  minilm_embeddings.npy     : {mini_embeddings.shape}")
if 'oai_embeddings' in dir():
    print(f"  openai_embeddings.npy     : {oai_embeddings.shape}")
if 'google_embeddings' in dir():
    print(f"  google_embeddings.npy     : {google_embeddings.shape}")
print(f"  {your_save_path:<28}: {your_embeddings.shape}")
print(f"\nTo load in Notebook 2:")
print(f"  embeddings = np.load('minilm_embeddings.npy')")
print(f"  page_ids = pd.read_csv('page_ids.csv')")

---
## Summary

| Model | Dimensions | File | Cost |
|-------|------------|------|------|
| all-MiniLM-L6-v2 | 384 | `minilm_embeddings.npy` | Free (local) |
| text-embedding-3-small | 1,536 | `openai_embeddings.npy` | ~$0.06 |
| text-embedding-004 | 768 | `google_embeddings.npy` | Free (API) |
| Your choice | varies | `{model_name}_embeddings.npy` | varies |

**What you have done:** Generated document embeddings (BERTopic Step 2) using multiple models, compared them, and verified they capture meaningful semantic structure.

**What is next (Notebook 2):** Take these embeddings and run the rest of the pipeline: UMAP dimensionality reduction, HDBSCAN clustering, and c-TF-IDF topic representation to discover topics and compare them against the curated human labels.